In [1]:
from datetime import datetime,timedelta

In [3]:
class Book:
    
    def __init__(self,title,author,isbn,total_copies):
        
        self.title= title
        self.author=author
        self.isbn=isbn
        self.total_copies= total_copies
        self.available_copies= total_copies
        self.reservations= []

    def borrow(self):
        if self.available_copies>0:
            self.available_copies-= 1
            return True
        else:
            return False
            
    def return_book(self):
        if self.available_copies < self.total_copies:
            self.available_copies+=1
    
    
    def reserve(self, member_id):
        self.reservations.append(member_id)
    
    def __str__(self):
        
        return f"Name: {self.title}\nISBN: {self.isbn}\nAvailable Copies: {self.available_copies}"

In [4]:
class Member:
    
    def __init__(self,member_id,name,email,max_books=3):
        
        self.member_id= member_id
        self.name= name
        self.email= email
        self.max_books = max_books
        self.borrowed_books= []
    
    def can_borrow(self):
        return len(self.borrowed_books) < self.max_books
    
    def borrow_book(self,book_isbn):
        
        if self.can_borrow():
            self.borrowed_books.append(book_isbn)
            
        else:
            raise ValueError("Member has reached borrowing limit")
        
    def return_book(self,book_isbn):
        
        if book_isbn in self.borrowed_books:
            self.borrowed_books.remove(book_isbn)
        
        else:
            raise ValueError("Book not borrowed by this member")
            
    def __str__(self):
        return f"Name: {self.name}"

In [5]:
class Loan:
    
    def __init__(self,book_isbn,member_id,loan_days=14):
        
        self.book_isbn= book_isbn
        self.member_id= member_id
        self.returned= False
        self.loan_date= datetime.now()
        self.due_date=  self.loan_date + timedelta(days=loan_days)
        
            
    def check_overdue(self):
        
        return not self.returned and datetime.now() > self.due_date
    
    def calculate_fine(self):
        
        if self.check_overdue():
            days_overdue= (datetime.now()-self.due_date).days
            return days_overdue*5
            
        else:
            return 0
            
    
    def return_loan(self):
        self.returned= True

In [6]:
class Library:
    
    def __init__(self):
        self.books= {}    #ISBN -> Book
        self.members= {}  #Member_Id -> Member 
        self.loans= []
        
        
    def add_book(self,title,author,isbn,copies=1):
        
        if isbn in self.books:
            self.books[isbn].total_copies += copies
            self.books[isbn].available_copies+= copies
            
        else:
            self.books[isbn] = Book(title,author,isbn,copies)
        
    def register_member(self,name,email,member_id):
        
        if member_id in self.members:
            raise ValueError("MEMBER ID already exists")
            
        else:
            self.members[member_id]= Member(member_id,name,email)
        
    
    def borrow_book(self,isbn,member_id):
        
        if isbn not in self.books:
            raise ValueError("Book Not Found")
            
        if member_id not in self.members:
            raise ValueError("Member Not Found")
            
        book= self.books[isbn]
        member= self.members[member_id]
        
        if not member.can_borrow():
            raise ValueError("Member reached borrowing limit")
            
        if not book.borrow():
            raise ValueError("No copies available")
            
        member.borrow_book(isbn)
        self.loans.append( Loan(isbn,member_id) )
        
    def return_book(self,isbn,member_id):        
        
        if isbn not in self.books:
            raise ValueError("Book Not Found")
            
        if member_id not in self.members:
            raise ValueError("Member Not Found")
        
        book= self.books[isbn]
        member= self.members[member_id]
        
        member.return_book(isbn)
        book.return_book()
        
        for loan in self.loans:
            
            if loan.book_isbn == isbn and loan.member_id==member_id and not loan.returned:
                loan.return_loan()
                break
    
    def get_overdue_loans(self):
        return [loan for loan in self.loans if self.check_overdue()]
            
    def get_member_loans(self,member_id):
        return [loan for loan in self.loans if loan.member_id==member_id]

In [7]:
library = Library()
# Add some books
library.add_book("The Great Gatsby", "F. Scott Fitzgerald", "9780743273565", 3)
library.add_book("To Kill a Mockingbird", "Harper Lee", "9780061120084", 2)
library.add_book("1984", "George Orwell", "9780451524935", 5)


# Register members
library.register_member("John Doe", "john@example.com", "M001")
library.register_member("Jane Smith", "jane@example.com", "M002")


# Borrow books
library.borrow_book("9780743273565", "M001")
library.borrow_book("9780061120084", "M001")
library.borrow_book("9780451524935", "M002")


# Display library status
# print("\nLibrary Books:")
# for book in library.books.values():
#     print(book)
        
print("\nLibrary Members:")
for member in library.members.values():
    print(member)        


Library Members:
Name: John Doe
Name: Jane Smith
